## Импорты

In [1]:
import pandas as pd
import numpy as np

## Читаем датасеты

In [2]:
age_ratings_df = pd.read_csv('data/age_ratings.csv')
org_df = pd.read_csv('data/age_rating_organizations.csv')
age_cat_df = pd.read_csv('data/age_rating_categories.csv')
df_decriptions = pd.read_csv('data/age_rating_content_descriptions_v2.csv')


## Мерджим

In [3]:
res = pd.merge(age_ratings_df, org_df, left_on = 'organization', right_on = 'id', how = 'left', suffixes = ('', '_org'))
res.head()


,id,organization,rating_category,rating_content_descriptions,id_org,name
0,3,1,6,"[3, 29]",1,ESRB
1,10,1,5,"[29, 3]",1,ESRB
2,23,1,3,NaN,1,ESRB
3,24,1,3,NaN,1,ESRB
4,25,1,3,NaN,1,ESRB


In [4]:

res = pd.merge(res, age_cat_df, left_on='rating_category', right_on='id', how='left', suffixes=('', '_cat'))
res = res[['id', 'name', 'rating_content_descriptions', 'rating']]
res['rating_content_descriptions'] = res['rating_content_descriptions'].str.replace('[', '').str.replace(']', '').str.split(',')
res.head()

,id,name,rating_content_descriptions,rating
0,3,ESRB,"[3, 29]",M
1,10,ESRB,"[29, 3]",T
2,23,ESRB,NaN,E
3,24,ESRB,NaN,E
4,25,ESRB,NaN,E


In [5]:

res = res.explode('rating_content_descriptions')
res['rating_content_descriptions'] = res['rating_content_descriptions'].astype('Int64')
res = pd.merge(res, df_decriptions, left_on = 'rating_content_descriptions', right_on = 'id', how = 'left', suffixes = ('', '_decr'))
res.head()


,id,name,rating_content_descriptions,rating,id_decr,description,organization
0,3,ESRB,3,M,3.0,Blood,1.0
1,3,ESRB,29,M,29.0,Violence,1.0
2,10,ESRB,29,T,29.0,Violence,1.0
3,10,ESRB,3,T,3.0,Blood,1.0
4,23,ESRB,<NA>,E,NaN,NaN,NaN


In [6]:

res = res[['id', 'name', 'description', 'rating']]
res['description'] = res['description'].fillna('no_description')
res = res.groupby(['id', 'name', 'rating'], as_index=False)['description'].agg(lambda x: ', '.join(x))
res.head()

,id,name,rating,description
0,3,ESRB,M,"Blood, Violence"
1,10,ESRB,T,"Violence, Blood"
2,23,ESRB,E,no_description
3,24,ESRB,E,no_description
4,25,ESRB,E,no_description


In [7]:
res.to_csv('data/age_ratings_merged.csv', index=False)

In [11]:
rating_age = age_cat_df.merge(org_df, left_on='organization', right_on='id', how='left').drop(columns = ['organization', 'id_y'])
rating_age

,id_x,rating,name
0,1,RP,ESRB
1,2,EC,ESRB
2,3,E,ESRB
3,4,E10+,ESRB
4,5,T,ESRB
5,6,M,ESRB
6,7,AO,ESRB
7,8,3,PEGI
8,9,7,PEGI
9,10,12,PEGI


In [ ]:
rating_age.to_excel('data/rating_age.xlsx', index = False)